In [2]:
import pandas as pd

In [3]:
eval_csv_path = r"knight_fintech_hr_rag_eval.csv"
q = pd.read_csv(eval_csv_path)

In [4]:
q.head()

,question,answer,difficulty,context
0,What is the effective date of HR Policy Handbo...,"June 1, 2024.",Easy,Cover page: Policy Versions table lists V2.1 e...
1,Which email address should employees use for g...,hr@knightfintech.com (Human Resource department).,Easy,Introduction contact table: Grievances / Perfo...
2,A Performance Improvement Plan lasts 30 days a...,The handbook is internally inconsistent. Secti...,Tricky,E.3.0: 'A PIP's duration shall be 30 days. if ...
3,How many hours per week are employees expected...,"At least 45 hours per week, across 5 days a we...",Easy,C.3.0 Working hours: 'Each employee is expecte...
4,Which software is used to mark attendance and ...,GreytHR (the company's payroll software / HRMS).,Easy,C.3.0: 'All the employees have to mark their a...


In [5]:
import asyncio
import httpx
import pandas as pd


async def update_csv(rag_url, df):
    df["results"] = None
    df["retrieval_context"] = None

    async def send_request(client, index, question, doc_id):
        payload = {"question": question, "doc_id": doc_id}
        try:
            res = await client.post(rag_url, json=payload)
            

            res.raise_for_status()
            data = res.json()
            answer = data.get("answer", "No answer field found")
            context = data.get("context", "No context field found")

            return index, answer, context
        except Exception as e:
            error_msg = f"Error: {str(e)}"
            print(f"Error at index {index} (doc_id {doc_id}): {error_msg}")
            return index, error_msg, error_msg

    async with httpx.AsyncClient(timeout=120.0) as client:
        tasks = []
        for i in range(len(df)):
            question = df.iloc[i]["question"]
            doc_id = "92348dc1-5aa"

            tasks.append(send_request(client, i, question, doc_id))

        completed_tasks = await asyncio.gather(*tasks)

    for index, answer, context in completed_tasks:
        df.at[index, "results"] = answer
        df.at[index, "retrieval_context"] = (
            str(context) if isinstance(context, (list, dict)) else context
        )

    return df


In [6]:
import httpx
import pandas as pd


def update_seq_csv(rag_url, df):
    df["results"] = None
    df["retrieval_context"] = None

    doc_id = "92348dc1-5aa"

    with httpx.Client(timeout=120.0) as client:
        for i in range(len(df)):
            question = df.iloc[i]["question"]
            payload = {"question": question, "doc_id": doc_id}

            try:
                res = client.post(rag_url, json=payload)
                res.raise_for_status()

                data = res.json()
                answer = data.get("answer", "No answer field found")
                context = data.get("context", "No context field found")

            except Exception as e:
                error_msg = f"Error: {str(e)}"
                print(f"Error at index {i} (doc_id {doc_id}): {error_msg}")
                answer = error_msg
                context = error_msg

            df.at[i, "results"] = answer
            df.at[i, "retrieval_context"] = (
                str(context) if isinstance(context, (list, dict)) else context
            )

    return df


In [7]:
# df =  await update_csv("http://127.0.0.1:1222/query", q)
df =   update_seq_csv("http://127.0.0.1:1222/query", q)

In [8]:
from llm_local import get_local_llm
from deepeval.models import DeepEvalBaseLLM


class AgentLLM(DeepEvalBaseLLM):
    def load_model(self):
        return get_local_llm()

    def generate(self, prompt: str) -> str:
        response = self.model.invoke(prompt)
        if not isinstance(response.content, str):
            raise TypeError("Expected the local LLM to return text content.")
        return response.content

    async def a_generate(self, prompt: str) -> str:
        response = await self.model.ainvoke(prompt)
        if not isinstance(response.content, str):
            raise TypeError("Expected the local LLM to return text content.")
        return response.content

    def get_model_name(self) -> str:
        return "Qwen2.5-14B-Instruct"


llm = AgentLLM()

[llm_local] Loading model from: E:\DS_Team_Share\hugging-face-model\Qwen2.5-14B-Instruct\models--Qwen--Qwen2.5-14B-Instruct\snapshots\cf98f3b3bbb457ad9e2bb7baf9a0125b6b88caa8


W1026 11:20:20.982000 11484 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

In [9]:

from deepeval.metrics import (
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric
)

contextual_precision = ContextualPrecisionMetric(model=llm, async_mode=False)
contextual_recall = ContextualRecallMetric(model=llm, async_mode=False)
contextual_relevancy = ContextualRelevancyMetric(model=llm, async_mode=False)


In [10]:
from deepeval.test_case import LLMTestCase
import csv

test_cases = []
# test_case = LLMTestCase(
#     input="",
#     actual_output="",
#     expected_output="",
#     retrieval_context=[
       
#     ]
# )
for _, row in q.iterrows():
    context_value = row.get("retrieval_context", row.get("context", []))
    retrieval_context = [part.strip() for part in context_value.split("\n\n") if part.strip()]
   
    test_cases.append(
        LLMTestCase(
            input=row.get("question", ""),
            actual_output=row.get("results", ""),
            expected_output=row.get("answer", ""),
            retrieval_context=retrieval_context,
        )
    )


In [11]:
def save_metric_results(evaluation_result):
    for evaluated_case in evaluation_result.test_results:
        row_index = evaluated_case.index
        for metric_result in evaluated_case.metrics_data:
            metric_name = metric_result.name.lower().replace(" ", "_")
            q.at[r.ow_index, f"{metric_name}_score"] = metric_result.score
            q.at[row_index, f"{metric_name}_passed"] = metric_result.success
            q.at[row_index, f"{metric_name}_reason"] = metric_result.reason
            q.at[row_index, f"{metric_name}_error"] = metric_result.error

    q.to_csv(f'{eval_csv_path}_metric.csv', index=False)
    print(f"Saved metric results to {eval_csv_path}")

In [ ]:
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig


config = AsyncConfig(run_async=False)
evaluation_result = evaluate(
    test_cases=test_cases,
    metrics=[contextual_precision, contextual_recall, contextual_relevancy],
    async_config=config
)
save_metric_results(evaluation_result)

: 

In [ ]:
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric
answer_relevancy = AnswerRelevancyMetric(model=llm, async_mode=False)
faithfulness = FaithfulnessMetric(model=llm, async_mode=False)

In [ ]:
from deepeval import evaluate
config = AsyncConfig(run_async=False)

evaluation_result = evaluate(
    test_cases=test_cases,
    metrics=[answer_relevancy, faithfulness],
    async_config=config
)
save_metric_results(evaluation_result)

In [ ]:
import sys
import torch
import transformers.modeling_utils
# Manually inject the model into transformers' root lookup map
sys.modules['transformers'].PreTrainedModel = transformers.modeling_utils.PreTrainedModel

# Your original imports continue below...
from rag_pipeline.rag_agent import RAGAgent
